<a href="https://colab.research.google.com/github/em2tech/Machine_Learning_SJU/blob/main/HW7/HW7_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1. Why is feature scaling essential prior to executing K-Means, DBSCAN, or HDBSCAN on the Wine dataset? Specifically, how would unscaled features distort distance calculations relative to other features? Provide specific examples.**



**Answer:**  Without scaling, clustering on the Wine data is effectively clustering on prline alone, because proline carries 99.77% of the total raw variance. All three algorithms in this assignment decide everything from Euclidean distance, so each feature's influence is proportional to its numeric spread, not to its chemical importance.

How distance depends on scacle
The squared Euclidean distance between two wines is a sum of per-feature terms:

d(a,b)^2 = \sum_{j=1}^{13} (a_j - b_j)^2

Each term is measured in that feature's own units. A feature whose values ediffer by hundreds contributes tens of thousands to the sum; a feature whose values differ by tenths contributes hundreths. The alorithm has no way to know that a 0.3 difference in hue may matter as much chemically as a 300 mg/L difference in prolinel.

**2. How did the Elbow Method and Silhouette Analysis in Task 2 reflect the natural structure of the Wine dataset? Did both evaluation metrics align with the true number of wine cultivars (k=3)? Explain any discrepancies observed.**



**Answer:** Both the Elbow Method and Silhouette Analysis pointed to k = 3, matching the three true wine cultivars. Inertia fell 28.3% from k = 1 to k = 2 and another 23.0% from k = 2 to k = 3, but only 8.0% from k = 3 to k = 4. Every step after that was between 4.7% and 6.2%. That sharp bend shows the first two splits capture real structure, and later splits only carve existing groups into smaller pieces. The objective max-distance-to-chord rule also selected k = 3, so the elbow choice was not based on eyeballing alone. The silhouette score peaked at k = 3 (0.285), meaning three clusters gave the best balance of cohesion and separation. The k = 3 solution was also correct in substance: it matched the true labels with an ARI of 0.897. Only 6 of 178 wines were misassigned, all from Class 1.
The evidence, however, is moderate rather than decisive. A silhouette of 0.285 indicates weak but real structure; well-separated clusters usually score above 0.5. The scores for k = 2 (0.259) and k = 4 (0.260) were close behind. The main reason is Class 1, the largest (71 wines) and most spread-out cultivar. It sits between Class 0 and Class 2 and overlaps both, so a two-way split is almost as coherent as a three-way one. Its borderline wines, such as #61 (distances of 3.52 vs. 3.40 to two centroids), pull the average silhouette down. Three other factors also limit the scores. In 13 dimensions, distances between points tend to be similar, which compresses the gap silhouette measures. Standardization gives weak features like ash and magnesium the same weight as strong separators like flavanoids and proline. And the elbow is a heuristic whose bend can be harder to locate on less structured data.
In short, the data contain three real groups, and both metrics found them. The modest scores reflect that one of those groups overlaps the other two.


**3. Compare the clustering behavior and cluster assignments of K-Means against DBSCAN and HDBSCAN on the 13-dimensional Wine feature space. Why did DBSCAN and HDBSCAN classify specific wine samples as noise, while K-Means assigned every sample to a cluster?**



**Answer:** K-Means split all 178 wines into three clusters that closely match the cultivars (ARI 0.897). It captured Class 0 and Class 2 completely and misplaced only 6 Class 1 wines. Both density-based methods found the same two dense cores instead: one mixing Class 0 with much of Class 1, and one that is almost purely Class 2. They labelled the sparse wines in between as noise. DBSCAN (eps 2.3, min_samples 3) merged 56 Class 0 and 44 Class 1 wines into one large cluster. It also formed a clean Class 2 cluster (42 of 43 wines) and a 3-wine micro-cluster, and labelled 32 wines as noise (ARI 0.399). HDBSCAN (min_cluster_size 5) produced smaller, purer cores; its Class 2 cluster is 97% pure. It declared 58 wines noise, including more than half of all Class 1 wines (37 of 71), for an ARI of 0.342. Neither method could separate Class 0 from Class 1. In 13-D space, a chain of moderately dense Class 1 wines connects to the Class 0 region, and the Task 7 dendrogram shows these groups merging below a distance of 2.3.

The difference comes from what each algorithm is designed to find. K-Means has no concept of noise. It minimizes squared distance to centroids, so the whole space becomes a Voronoi partition, and every wine is assigned to its nearest centroid, however far away it is. Wine #121, for example, is 6.2–7.5 units from all three centroids, over four times farther than typical prototype wines. K-Means still assigns it to a cluster, even though its soft profile (0.40 / 0.13 / 0.47) shows it belongs nowhere. DBSCAN and HDBSCAN define clusters as dense regions. A wine joins a cluster only if it is a core point or lies within reach of one; otherwise it is labelled noise. The noise wines here are in sparse regions of 13-D space. In DBSCAN, 49 wines fail the core test, and 32 of them have no core point within reach. Most noise wines sit in the low-density gap between cultivars, which is why Class 1 makes up most of the noise (23 of 32 for DBSCAN, 37 of 58 for HDBSCAN). Others are extreme on one or two chemicals, such as magnesium at z = +4.4. The general sparsity of 178 points in 13 dimensions adds to this. Some noise points even look central on the 2-D PCA plots, because PCA keeps only 55.4% of the variance. HDBSCAN flagged more noise than DBSCAN because it requires five neighbours rather than three and keeps only clusters that persist across many density levels. Its noise count ranged from 3 to 82 as min_cluster_size went from 3 to 15.

For recovering the cultivars, K-Means clearly performed best. Its assumptions of a fixed number of roughly spherical, similar-sized groups, with every point assigned, happen to fit this dataset. DBSCAN and HDBSCAN make no shape assumptions and refuse to force sparse wines into groups. That is useful for detecting outliers, and their noise labels correctly flag the ambiguous between-cultivar wines. But on a small, high-dimensional dataset with overlapping classes, density is too weak a signal to separate Class 0 from Class 1.

**4. Contrast the operational mechanisms of standard DBSCAN and HDBSCAN. How does HDBSCAN's hierarchical, stability-based approach overcome DBSCAN's vulnerability to a single global eps parameter when clustering chemical datasets with varying local densities?**



**Answer:** DBSCAN clusters the data at a single density level set by the user, while HDBSCAN builds clusterings at every density level and keeps the clusters that last longest. In DBSCAN, a wine with at least min_samples neighbours within radius eps is a core point. Core points within eps of each other form a cluster, non-core wines within reach of a core point become border points, and everything else is noise. Because one eps defines "dense" for the whole dataset, DBSCAN is effectively cutting a single-linkage tree with one horizontal line at height eps. On the Wine data, that proved fragile. With min_samples = 3, eps = 1.8 produced 8 fragmented clusters with half the wines labelled noise. Eps = 2.3 gave two real groups plus a 3-wine micro-cluster, and eps = 2.4 gave two groups. At eps = 2.5, everything merged into one cluster (ARI ≈ 0). The cause is varying local density. The Class 0 and Class 2 cores are dense, while the Class 1 wines around and between them are sparse. Any eps small enough to keep the dense cores apart leaves the sparse wines as noise. Any eps large enough to absorb them bridges the cores and merges them. No single radius works for both, and the Task 7 cut analysis confirmed that no single cut of the tree yields three clean groups.

HDBSCAN avoids this problem in several steps. It first computes each wine's core distance, the distance to its 5th nearest neighbour, which is small in dense regions and large in sparse ones. It then replaces each pairwise distance with the mutual reachability distance: the largest of the two wines' core distances and their true distance. This leaves dense regions unchanged but pushes sparse wines away, so thin bridges become expensive to cross. A single-linkage hierarchy built on these distances is equivalent to running DBSCAN at every eps at once. HDBSCAN then condenses this tree, treating any split smaller than min_cluster_size as points falling out of a cluster rather than a new cluster. Finally, it scores each cluster's stability, meaning how many wines it holds and over how wide a range of densities. It keeps the most stable set of non-overlapping clusters, labels the remaining wines as noise, and gives every wine a membership strength between 0 and 1.

This approach overcomes the single-eps problem in four ways. Different clusters can be selected at different density levels, so a dense core and a looser group can coexist. There is no eps to tune, only min_cluster_size, which is far more intuitive. HDBSCAN found 2 clusters for every min_cluster_size from 3 to 15, while DBSCAN's cluster count ranged from 0 to 9 across its grid. Mutual reachability makes it harder for dense cores to chain together through a few in-between points. And membership strengths (averaging 0.95 and 0.97 in the two clusters) show which wines are central and which are peripheral.

HDBSCAN still could not separate Class 0 from Class 1. With only 178 wines in 13 dimensions, Class 1 does not form its own density peak. It forms a shoulder on the Class 0 peak, and a density-based method cannot isolate a group that is not its own peak. On this dataset, HDBSCAN's advantage over DBSCAN was stability and ease of tuning, not a better match to the cultivars (ARI 0.342 vs. 0.399).

**5.Based on your comparative classification experiment in Task 8, how effectively did the cluster assignments from K-Means, DBSCAN, and HDBSCAN enable a downstream classifier to predict the true ground-truth wine cultivars (Class 0-2)? Discuss the limitations of mapping unsupervised clusters to supervised class labels when dealing with noise and density-based groupings.**

**Answer:** K-Means cluster labels alone predicted the correct cultivar for 35 of 36 test wines (97.2%), compared with 75.0% for HDBSCAN and 72.2% for DBSCAN. All three beat the 38.9% majority-class baseline, but the density-based methods fell far below K-Means. In this experiment, each algorithm's cluster labels were one-hot encoded, with noise kept as its own category. A 300-tree Random Forest used only those labels to predict the cultivar on the same stratified 80/20 split (142 training and 36 test wines). For comparison, a Random Forest trained on all 13 scaled features scored 100%. Because every wine in a cluster has the same input, the classifier can only learn a lookup from each cluster to its most common cultivar, so accuracy is limited by cluster purity, not by the classifier. K-Means clusters were nearly pure (94–100%), so it made only one error. DBSCAN's large cluster held 56 Class 0 and 44 Class 1 wines and was mapped to Class 0, which misclassified 9 of the 14 Class 1 test wines. HDBSCAN's mixed cluster cost it 5 Class 1 wines. Its noise group, however, was 64% Class 1, so "noise" was mapped to Class 1. That correctly recovered 9 Class 1 test wines but misclassified 4 noise wines from other classes. Because Class 1 is the sparse, in-between cultivar, the noise label acted as a weak proxy for it, which is why HDBSCAN slightly outperformed DBSCAN.

The silhouette scores did not track accuracy. HDBSCAN had the highest silhouette (0.352), but it was calculated only on the 120 dense, well-separated wines it kept, after its 58 hardest wines were excluded as noise. K-Means (0.285) was scored on all 178 wines, including every borderline case. Silhouette rewards compact, separated groups, not agreement with the true classes, so a method that discards difficult points will look better on this measure.

The experiment also shows the limitations of using unsupervised clusters as class labels. When fewer clusters are found than there are classes, as with DBSCAN and HDBSCAN finding two groups for three cultivars, at least one class is always misclassified, and no downstream model can recover a distinction the clustering erased. Noise is not a real class: it mixes wines from all three cultivars that share only low density. It helped here by chance and could add only confusion on other data. Dropping noise wines would raise reported accuracy but leave 18–33% of wines without a prediction. Density-based clusters reflect the data's shape, and nothing guarantees that a density peak corresponds to a grape variety. Density-based labels are also unstable: HDBSCAN's noise count ranged from 3 to 82 as min_cluster_size changed. Unlike K-Means, DBSCAN and HDBSCAN cannot assign new wines to existing clusters. In this experiment, the test wines therefore helped shape the clusters, which a real deployment could not do. One-hot labels also discard information about how confidently a wine belongs to a cluster; centroid distances or membership strengths would preserve it. Finally, with only 36 test wines, each error shifts accuracy by 2.8 points. The one-wine gap between DBSCAN and HDBSCAN is not meaningful, but the 9–10-wine gap to K-Means is.

Overall, centroid-based clustering recovered the natural groups in the Wine Recognition data, while density-based clustering did not, because the three cultivars form overlapping clouds of similar size rather than separate density peaks. Feature scaling was essential, since without it proline would have accounted for 99.77% of the raw variance and dominated every distance. The elbow method and silhouette analysis both selected k = 3, and that solution matched the cultivars with an ARI of 0.897. DBSCAN and HDBSCAN each found two dense cores and labelled the sparse, mostly Class 1 wines between them as noise. That is useful for outlier detection but cost them the Class 0 vs. Class 1 separation. HDBSCAN was far less sensitive to tuning than DBSCAN, but neither could create a cluster where no separate density peak exists. As pseudo-labels for classification, K-Means clusters achieved 97.2% accuracy, while DBSCAN and HDBSCAN reached 72–75%, limited by cluster purity rather than by the classifier.